In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Install audio and Korean NLP tools
#!pip install librosa soundfile torch torchaudio
!pip install g2pk  # This is the Grapheme-to-Phoneme tool mentioned in your report

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 579.6/579.6 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.0/438.0 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.5/34.5 MB 24.9 MB/s eta 0:00:00


In [ ]:
import json
import pandas as pd
import os
import librosa
import soundfile as sf
from tqdm import tqdm
from g2pk import G2p
from sklearn.model_selection import train_test_split
import torch
import torchaudio
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor, Trainer, TrainingArguments
from torch.utils.data import Dataset

from transformers import Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor
import torch
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Union

[nltk_data] Downloading package cmudict to /root/nltk_data...
[nltk_data]   Unzipping corpora/cmudict.zip.


In [ ]:
# Define path
dataset_path = '/content/drive/MyDrive/manual_datasets/clovacall_data/'
json_file = os.path.join(dataset_path, 'train_ClovaCall.json')

# 1. Load the JSON file correctly
with open(json_file, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

# 2. Convert to DataFrame
# This handles the list of records format you showed earlier
metadata = pd.DataFrame(raw_data)

# 3. Quick Clean: Remove the .1, .2 artifacts from the text
metadata['text'] = metadata['text'].str.replace(r'\.\d+$', '', regex=True)

# Preview the fixed data
print(f"Total Rows: {len(metadata)}")
print(metadata.head())

Total Rows: 59662
                          wav    text speaker_id
0  44_0513_936_0_03913_05.wav   김은지요.      03913
1  41_0531_954_0_08493_09.wav    7시요.      08493
2  41_0608_946_0_07631_09.wav   3명이요.      07631
3  42_0524_948_0_00422_00.wav   두명이요.      00422
4  42_0531_845_0_01679_02.wav  예약 돼요?      01679


In [ ]:
import librosa
import soundfile as sf
from tqdm import tqdm

source_dir = '/content/drive/MyDrive/manual_datasets/clovacall_data/wavs_train/'
target_dir = '/content/drive/MyDrive/manual_datasets/clovacall_data/wavs_resampled_16k/'

if not os.path.exists(target_dir):
    os.makedirs(target_dir)

In [ ]:
def resample_all_files(metadata_df, source_path, target_path, batch_size=10000):
    total_files = len(metadata_df)
    print(f"Starting full automation for {total_files} files...")

    # Loop through the entire dataframe in steps of batch_size
    for start_idx in range(0, total_files, batch_size):
        end_idx = min(start_idx + batch_size, total_files)
        batch_df = metadata_df.iloc[start_idx:end_idx]

        print(f"\n--- Processing Global Range: {start_idx} to {end_idx} ---")

        for filename in tqdm(batch_df['wav'], desc=f"Batch {start_idx//batch_size + 1}"):
            input_file = os.path.join(source_path, filename)
            output_file = os.path.join(target_path, filename)

            # Resume logic: skip if already resampled
            if os.path.exists(output_file):
                continue

            try:
                # Wav2Vec 2.0 strictly requires 16,000 Hz [cite: 191]
                audio, sr = librosa.load(input_file, sr=16000)
                sf.write(output_file, audio, 16000)
            except Exception as e:
                print(f"Error processing {filename}: {e}")

# --- SETTINGS ---
# No need to manually update CURRENT_START anymore!
resample_all_files(metadata, source_dir, target_dir, batch_size=10000)

Starting full automation for 59662 files...

--- Processing Global Range: 0 to 10000 ---


Batch 1: 100%|██████████| 10000/10000 [03:37<00:00, 45.92it/s] 



--- Processing Global Range: 10000 to 20000 ---


Batch 2: 100%|██████████| 10000/10000 [00:01<00:00, 5304.67it/s]



--- Processing Global Range: 20000 to 30000 ---


...

Batch 4: 100%|██████████| 10000/10000 [00:01<00:00, 6457.22it/s]



--- Processing Global Range: 40000 to 50000 ---


Batch 5: 100%|██████████| 10000/10000 [00:01<00:00, 5779.20it/s]



--- Processing Global Range: 50000 to 59662 ---


Batch 6: 100%|██████████| 9662/9662 [00:01<00:00, 5285.57it/s]


In [ ]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

In [ ]:
from g2pk import G2p

# Initialize the G2P tool for phonetic labels
# This is your "Gold Standard" for calculating Phoneme Error Rate (PER) [cite: 196, 197]
g2p = G2p()

def process_everything_automated(metadata_df, source_path, target_path, batch_size=5000):
    total = len(metadata_df)

    # Ensure columns exist for your metadata update
    if 'phonetic_transcript' not in metadata_df.columns:
        metadata_df['phonetic_transcript'] = ""

    for start_idx in range(0, total, batch_size):
        end_idx = min(start_idx + batch_size, total)
        print(f"\n--- Processing Global Range: {start_idx} to {end_idx} ---")

        for i in tqdm(range(start_idx, end_idx)):
            row = metadata_df.iloc[i]
            filename = row['wav']
            text = row['text']

            input_file = os.path.join(source_path, filename)
            output_file = os.path.join(target_path, filename)

            # 1. Resample to 16kHz for Wav2Vec 2.0 [cite: 191]
            if not os.path.exists(output_file):
                try:
                    audio, sr = librosa.load(input_file, sr=16000)
                    sf.write(output_file, audio, 16000)
                except Exception:
                    continue # Skip broken audio files

            # 2. Generate Phonetic Labels [cite: 196, 211]
            try:
                # Convert "합니다" -> "함니다"
                metadata_df.at[i, 'phonetic_transcript'] = g2p(text)
            except Exception as e:
                print(f"G2P error at index {i}: {e}")

        # Save progress to Drive after every batch
        metadata_df.to_csv('/content/drive/MyDrive/clovacall_processed_metadata.csv', index=False)

# Run the full automation
process_everything_automated(metadata, source_dir, target_dir)


--- Processing Global Range: 0 to 5000 ---


100%|██████████| 5000/5000 [02:29<00:00, 33.47it/s]



--- Processing Global Range: 5000 to 10000 ---


100%|██████████| 5000/5000 [01:56<00:00, 42.97it/s]



--- Processing Global Range: 10000 to 15000 ---


...

100%|██████████| 5000/5000 [01:57<00:00, 42.50it/s]



--- Processing Global Range: 50000 to 55000 ---


100%|██████████| 5000/5000 [01:57<00:00, 42.44it/s]



--- Processing Global Range: 55000 to 59662 ---


100%|██████████| 4662/4662 [01:50<00:00, 42.15it/s]


In [ ]:
from sklearn.model_selection import train_test_split

# 1. First, split into Train and a temporary "Test/Val" set
train_df, temp_df = train_test_split(metadata, test_size=0.2, random_state=42)

# 2. Split the temporary set equally into Validation and Test
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Test samples: {len(test_df)}")

Training samples: 47729
Validation samples: 5966
Test samples: 5967
